In [7]:
import pandas as pd
import requests
import json
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon 



In [8]:
demographics = json.loads(requests.get('https://files.jcrayb.com/files/ie300/majority.json').text)

In [9]:
centroids = {}

chicago = gpd.read_file('data/zips.geojson').dropna(subset='zip')[['zip', 'geometry']]
tracts = gpd.read_file('osmnx/data/tracts.geojson')



tract_demographics = {}

zips = chicago.zip.to_list()

zips = [zip for zip in zips if zip]

demographics = {zip: majority for zip, majority in demographics.items() if zip in zips}

for tract in tracts.iloc:
    centroid = tract.geometry.centroid

    for row in chicago.iloc:
        zip_geometry = row.geometry

        if zip_geometry.contains(centroid) and (row.zip in demographics):
            tract_demographics[tract.namelsad10] = demographics[row.zip]
            continue


In [10]:
json.dump(tract_demographics, open('data/tract_demographics.json', 'w'))

In [14]:
query = 'high-school'

destination = json.load(open(f'computation_results/final_results/{query}_final_results.json', 'r'))

res = {}

res_by_race = {

}



for tract, race in tract_demographics.items():
    n_cameras = []
    unrestricted_time_to_reach = []
    restricted_time_to_reach = []

    for hospital, results in destination[tract].items():
        budgets = list(results.keys())
        highest, lowest = (budgets[0], budgets[-1])
        #print(highest, lowest)

        n_cameras += [int(highest)]
        unrestricted_time_to_reach += [results[highest]]
        restricted_time_to_reach += [results[lowest]]
    
    mean_uttr = np.mean(unrestricted_time_to_reach)
    mean_rttr = np.mean(restricted_time_to_reach)

    res[tract] = {
        'demographic majority': race,
        'Avg time to reach destination': mean_uttr,
        'Avg time to reach destination while avoiding cameras': mean_rttr,
        '% difference in time': (mean_rttr-mean_uttr)/mean_uttr*100,
        'absolute difference in time': mean_rttr-mean_uttr,
        'Avg n cameras': np.mean(n_cameras)
    }



for tract, results in res.items():
    metrics = list(results.keys())
    dem_maj = results['demographic majority']

    if not dem_maj in res_by_race:
        res_by_race[dem_maj] = {metric: [] for metric in metrics if metric != 'demographic majority'}

    for metric in metrics:
        if metric == 'demographic majority':
            continue
        res_by_race[dem_maj][metric] += [results[metric]]

total_results = {}

for race, res in res_by_race.items():
    total_results[race] = {}
    for metric, values in res.items():
        total_results[race][metric] = (np.mean(values), np.std(values))

json.dump(total_results, open(f'./analysis/{query}.json', 'w'), indent = 2)

In [112]:
total_results

{'black': {'Avg time to reach destination': 824.3639670591363,
  'Avg time to reach destination while avoiding cameras': 852.8182276620697,
  '% difference in time': 0.0354895450972817,
  'absolute difference in time': 28.45426060293337,
  'Avg n cameras': 1.2389157832191888},
 'white': {'Avg time to reach destination': 743.9999773094596,
  'Avg time to reach destination while avoiding cameras': 787.6389687723249,
  '% difference in time': 0.059355610938969974,
  'absolute difference in time': 43.63899146286524,
  'Avg n cameras': 2.052083502024882}}